# Homework 3

In [1]:
import sys
sys.path.append('/Users/jonathancontreras/Desktop/College Folder/Senior Year/Math485/Unit 1 Tabular')

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

import dhs_util
from dhs_util import *

os.chdir('/Users/jonathancontreras/Desktop/College Folder/Senior Year/Math485/Unit 1 Tabular')
df = pd.read_csv('dhs_service_records_synthesized_final.csv')

df = dhs_preprocessing(df)
df, service_map = add_service_label(df)
df = add_age_bin(df)

recipient = get_recipient_attribute(df)

In [3]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.preprocessing import *
from mlxtend.frequent_patterns import association_rules
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpmax
from mlxtend.frequent_patterns import hmine


In [4]:
df_id_serv = get_id_service_matrix(df) # this gives number of times the service is used
df_id_serv.iloc[:,1:23] = df_id_serv.iloc[:,1:23] > 0  # this converts value into True or False
df_id_serv


serv,id,S01,S03,S04,S05,S02,S06,S07,S08,S09,...,S13,S14,S15,S16,S17,S18,S19,S20,S21,S22
0,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,3,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,4,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,5,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533794,535604,False,False,False,False,False,False,False,False,True,...,False,True,False,False,False,False,False,False,False,False
533795,535605,False,False,False,False,False,False,False,False,True,...,False,True,False,False,False,False,False,False,False,False
533796,535606,False,False,False,False,False,False,False,False,True,...,False,True,False,False,False,False,False,False,False,False
533797,535607,False,False,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,True,False


In [5]:
# prep list of services again
serv_list = []
for groups in df.groupby('id').groups.values():
    serv_list.append(df.loc[groups]['serv'].tolist())

# following the tutorial example 
def oneHotCoding(serv_list):
    te = TransactionEncoder()
    te_ary = te.fit(serv_list).transform(serv_list)
    te_df = pd.DataFrame(te_ary, columns=te.columns_)
    return te_df

serv_oneHot = oneHotCoding(serv_list)

In [6]:
min_support = .01
freq_itemset_apriori = apriori(serv_oneHot,min_support=min_support,use_colnames=True) # use apriori algorithm
freq_itemset_fpgrowth = fpgrowth(serv_oneHot,min_support=min_support,use_colnames=True) # use fp growth algorithm
def serv_rules(freq_itemsets,metrics,threshold):
    asso_rules = association_rules(freq_itemsets, metric=metrics, min_threshold=threshold)
    return asso_rules.sort_values(by='lift', ascending=False)[['antecedents', 'consequents', 'support', 'confidence', 'lift']]



# 1) extract service itemsets that have support greater than 0.01using apriori

In [7]:
apriori(df_id_serv.iloc[:,1:23], use_colnames=True, min_support=0.01)\
    .sort_values(by="support", ascending=False)

,support,itemsets
4,0.941422,(S12)
2,0.153844,(S09)
14,0.139131,"(S12, S09)"
15,0.103528,"(S14, S09)"
6,0.103528,(S14)
25,0.094436,"(S12, S14, S09)"
20,0.094436,"(S12, S14)"
3,0.040882,(S11)
18,0.032106,"(S12, S11)"
10,0.031396,(S19)


## 2) extract service itemsets that have support greater than 0.0001 using fpgrowth

In [8]:
fpmax(serv_oneHot,min_support=0.0001,use_colnames=True) #.sort_values(by="support", ascending=False).head(20) # determines what items are "frequent" given a min support

,support,itemsets
0,0.000199,"(S08, S12)"
1,0.000305,"(S12, S01)"
2,0.000124,"(S22, S12)"
3,0.000133,"(S12, S04, S02, S03)"
4,0.000131,"(S03, S09, S12, S14, S04)"
...,...,...
59,0.000122,"(S11, S09, S18, S12, S21, S14)"
60,0.000210,"(S09, S18, S12, S21, S14, S15)"
61,0.000204,"(S09, S18, S12, S19, S14)"
62,0.000169,"(S09, S12, S19, S14, S15)"


# 3) extract service itemsets that have support greater than 0.0001 using hmine

In [9]:
hmine(serv_oneHot,min_support=0.0001,use_colnames=True) #.sort_values(by="support", ascending=False).head(20)

,support,itemsets
0,0.000654,(S01)
1,0.000305,"(S12, S01)"
2,0.004191,(S02)
3,0.004142,"(S02, S03)"
4,0.000146,"(S04, S02, S03)"
...,...,...
930,0.000141,"(S21, S19)"
931,0.002471,(S20)
932,0.000126,"(S21, S20)"
933,0.01326,(S21)


# 4) make observations of the difference between them

a) We notice that itemsets like (S01, S12), (S09, S12), and (S09, S14) have the same support between the different algorithms. 

b) The hmine algorithm yields much better and more results, meaning that the dataset is sparse rather than dense i.e. recipients use less services rather than more. 

c) The itemsets returned by hmine and apriori are similar but there are fewer itemsets of cardinality 3 given by hmine than apriori

# 5) extract service itemsets that have support greater than 0.0001 using fpmax

In [10]:
fpmax(serv_oneHot,min_support=0.0001,use_colnames=True).sort_values(by="support", ascending=False)

,support,itemsets
56,0.000326,"(S09, S18, S12, S17, S14, S15)"
1,0.000305,"(S12, S01)"
34,0.000253,"(S11, S09, S18, S12, S16, S14, S07)"
63,0.000247,"(S11, S09, S12, S19, S14)"
11,0.000223,"(S02, S03, S09, S18, S12, S14)"
...,...,...
16,0.000101,"(S07, S10, S11)"
15,0.000101,"(S19, S10)"
29,0.000101,"(S11, S06, S09, S12, S21, S14, S15)"
17,0.000101,"(S10, S17, S09)"


FPmax finds the maximal frequent itemsets, the largest frequent itemsets (governed by min_support) that have no frequent supersets. Obviously, this means that the itemsets found by FPmax will be larger in cardinality than those found with the other algorithms, which aligns what we observe. FPmax also shows us that S12, S14, and S15 are often members of itemsets, which is not the immediate conclusion drawn from the previous algorithms.

# 6) extract length-1 service itemsets that have support greater than 0.01 using apriori


In [11]:
apri_1 = apriori(df_id_serv.iloc[:,1:23], use_colnames=True, min_support=0.01) #grab data frame 
apri_1[apri_1['itemsets'].apply(lambda x: len(x) == 1)] # apply len == 1 

,support,itemsets
0,0.013687,(S03)
1,0.012649,(S05)
2,0.153844,(S09)
3,0.040882,(S11)
4,0.941422,(S12)
5,0.011915,(S13)
6,0.103528,(S14)
7,0.024002,(S15)
8,0.012967,(S17)
9,0.013573,(S18)


# 7) extract length-2 service itemsets that have support greater than 0.005 using hmine

In [12]:
hmine_2 = hmine(serv_oneHot,min_support=0.005,use_colnames=True)
hmine_2[hmine_2['itemsets'].apply(lambda x: len(x) ==2)]

,support,itemsets
1,0.005274,"(S03, S09)"
3,0.012284,"(S12, S03)"
5,0.008057,"(S12, S05)"
8,0.006398,"(S07, S11)"
10,0.005105,"(S12, S07)"
12,0.018307,"(S11, S09)"
16,0.139131,"(S12, S09)"
27,0.008419,"(S13, S09)"
28,0.103528,"(S14, S09)"
32,0.019431,"(S15, S09)"


# 8) extract length-5 service itemsets that have support greater than 0.0001using fpmax

In [13]:
fpm5 = fpmax(serv_oneHot,min_support=0.0001,use_colnames=True).sort_values(by="support", ascending=False)
fpm5[fpm5['itemsets'].apply(lambda x: len(x) == 5)]

,support,itemsets
63,0.000247,"(S11, S09, S12, S19, S14)"
22,0.000208,"(S10, S09, S12, S14, S15)"
61,0.000204,"(S09, S18, S12, S19, S14)"
24,0.000197,"(S10, S11, S09, S12, S14)"
62,0.000169,"(S09, S12, S19, S14, S15)"
18,0.000163,"(S10, S09, S18, S12, S14)"
19,0.000157,"(S10, S09, S06, S12, S21)"
47,0.000150,"(S09, S12, S14, S15, S13)"
49,0.000150,"(S11, S09, S12, S14, S13)"
38,0.000142,"(S09, S12, S16, S19, S14)"


# 9) extract association rules that have lift greater than 0.5, using itemsets from fpgrowth with support greater than 0.05

In [14]:
fpg9 = fpgrowth(serv_oneHot,min_support=0.05,use_colnames=True)
serv_rules(fpg9,"lift",0.5)

,antecedents,consequents,support,confidence,lift
7,"(S12, S09)",(S14),0.094436,0.678758,6.556292
10,(S14),"(S12, S09)",0.094436,0.912184,6.556292
2,(S14),(S09),0.103528,1.000000,6.500073
3,(S09),(S14),0.103528,0.672938,6.500073
6,"(S12, S14)",(S09),0.094436,1.000000,6.500073
11,(S09),"(S12, S14)",0.094436,0.613843,6.500073
4,(S12),(S14),0.094436,0.100312,0.968942
5,(S14),(S12),0.094436,0.912184,0.968942
8,"(S14, S09)",(S12),0.094436,0.912184,0.968942
9,(S12),"(S14, S09)",0.094436,0.100312,0.968942


# 10) extract association rules that have support greater than 0.0003, using itemsets from fpmax with support greater than 0.0001


In [15]:
fpm10 = fpmax(serv_oneHot,min_support=0.0001,use_colnames=True)
asso_rules = association_rules(fpm10, metric="support", min_threshold=0.0003,support_only=True)
asso_rules.sort_values(by='lift', ascending=False)[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

,antecedents,consequents,support,confidence,lift
0,(S12),(S01),0.000305,NaN,NaN
1,(S01),(S12),0.000305,NaN,NaN
2,"(S09, S18, S12, S17, S14)",(S15),0.000326,NaN,NaN
3,"(S09, S18, S12, S17, S15)",(S14),0.000326,NaN,NaN
4,"(S09, S18, S12, S14, S15)",(S17),0.000326,NaN,NaN
...,...,...,...,...,...
59,(S18),"(S09, S12, S17, S14, S15)",0.000326,NaN,NaN
60,(S12),"(S09, S18, S17, S14, S15)",0.000326,NaN,NaN
61,(S17),"(S09, S18, S12, S14, S15)",0.000326,NaN,NaN
62,(S14),"(S09, S18, S12, S17, S15)",0.000326,NaN,NaN


# 11) if you were to make suggestions for DHS to consider increase the offering of one service, say, “Families_Receiving_Child_Welfare_Services” (S06), what other services would you suggest to offer together with? 



In [16]:
def predict(antecedent, rules, consequents_only = False):
    # get the rules for this antecedent
    preds = rules[rules['antecedents'] == antecedent]
    if consequents_only:
        # a way to convert a frozen set with one element to string
        preds = preds['consequents'].apply(iter).apply(next)
    return preds

# Try Minimum Support 0.001

In [18]:
min_support = 0.001

In [19]:
ap3 = apriori(serv_oneHot, min_support=min_support,use_colnames=True).sort_values(by="support", ascending=False)
ap3_rule = serv_rules(ap3,"confidence",0.50)
predict({'S06'}, ap3_rule, consequents_only=False)

,antecedents,consequents,support,confidence,lift
136,(S06),"(S21, S12)",0.003215,0.550353,62.201518
94,(S06),(S21),0.004582,0.784477,59.162639
127,(S06),(S12),0.003439,0.588839,0.625478


In [20]:
fpg3 = fpgrowth(serv_oneHot,min_support=min_support,use_colnames=True)
fpg3_rule = serv_rules(fpg3,"confidence",0.50)
predict({'S06'}, fpg3_rule, consequents_only=False)

,antecedents,consequents,support,confidence,lift
179,(S06),"(S21, S12)",0.003215,0.550353,62.201518
175,(S06),(S21),0.004582,0.784477,59.162639
176,(S06),(S12),0.003439,0.588839,0.625478


In [21]:
fpm3 = fpmax(serv_oneHot,min_support=min_support,use_colnames=True)
fpm3_rule = association_rules(fpm3, metric="confidence", min_threshold=0.00010, support_only=True)
predict({'S06'}, fpm3_rule, consequents_only=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
74,(S06),"(S21, S12, S14, S09)",NaN,NaN,0.001102,NaN,NaN,NaN,NaN,NaN


In [22]:
h3 = hmine(serv_oneHot,min_support=min_support,use_colnames=True)
h3_rule = association_rules(h3, metric="confidence", min_threshold=0.50, support_only=True)
predict({'S06'}, h3_rule, consequents_only=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric


# Try Minimum Support 0.0001

In [23]:
min_support = 0.0001

In [24]:
ap4 = apriori(serv_oneHot, min_support=min_support,use_colnames=True).sort_values(by="support", ascending=False)
ap4_rule = serv_rules(ap4,"confidence",0.50)
predict({'S06'}, ap4_rule, consequents_only=False)

,antecedents,consequents,support,confidence,lift
136,(S06),"(S21, S12)",0.003215,0.550353,62.201518
94,(S06),(S21),0.004582,0.784477,59.162639
127,(S06),(S12),0.003439,0.588839,0.625478


In [25]:
fpg4 = fpgrowth(serv_oneHot,min_support=min_support,use_colnames=True)
fpg4_rule = serv_rules(fpg4,"confidence",0.50)
predict({'S06'}, fpg4_rule, consequents_only=False)

,antecedents,consequents,support,confidence,lift
995,(S06),"(S21, S12)",0.003215,0.550353,62.201518
991,(S06),(S21),0.004582,0.784477,59.162639
992,(S06),(S12),0.003439,0.588839,0.625478


In [26]:
fpm4 = fpmax(serv_oneHot,min_support=min_support,use_colnames=True)
fpm4_rule = association_rules(fpm4, metric="confidence", min_threshold=0.00010, support_only=True)
predict({'S06'}, fpm4_rule, consequents_only=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
401,(S06),"(S21, S12, S10, S09)",NaN,NaN,0.000157,NaN,NaN,NaN,NaN,NaN
564,(S06),"(S21, S14, S16, S09)",NaN,NaN,0.000103,NaN,NaN,NaN,NaN,NaN
625,(S06),"(S09, S18, S12, S21, S14)",NaN,NaN,0.000146,NaN,NaN,NaN,NaN,NaN
687,(S06),"(S09, S12, S17, S21, S14)",NaN,NaN,0.000105,NaN,NaN,NaN,NaN,NaN
749,(S06),"(S09, S12, S17, S21, S15)",NaN,NaN,0.000114,NaN,NaN,NaN,NaN,NaN
874,(S06),"(S09, S11, S12, S21, S14, S15)",NaN,NaN,0.000101,NaN,NaN,NaN,NaN,NaN


In [27]:
h4 = hmine(serv_oneHot,min_support=min_support,use_colnames=True)
h4_rule = association_rules(h4, metric="confidence", min_threshold=0.50, support_only=True)
predict({'S06'}, h4_rule, consequents_only=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric


# Conclusion

Given our findings, I would confidently suggest to offer “Families_Receiving_Child_Welfare_Services” or S06 with S21, 'Parents_Receiving_Child_Welfare_Services', and S12, 'Individuals_Receiving_Income_Supports'.